In [ ]:
%matplotlib inline

import numpy as np
import pandas as pd
import pickle
import matplotlib.pyplot as plt

from quick_pp.database.objects import Project
from quick_pp.database.db_connector import DBConnector

db_conn = DBConnector()

# Load well from saved file
project_name = "30-7a"
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    all_data = project.get_all_data()

# Filter to sandshale only
all_data = all_data[all_data["model"] == "sandshale"].copy()

***
# Rock Typing

In [ ]:
from quick_pp.rock_type import (
    plot_cumulative_probability,
    plot_modified_lorenz,
    plot_fzi_histogram,
    plot_fzi_log_log,
    cluster_fzi,
    plot_fzi,
)
import json

clean_core_data = all_data.dropna(subset=["CPORE", "CPERM"]).copy()

fzi_cutoffs = [0.13, 0.4, 1, 2.7]
plot_fzi_log_log(clean_core_data.CPORE, clean_core_data.CPERM, cut_offs=fzi_cutoffs)

log_fzi_cutoffs = [round(np.log10(i), 3) for i in fzi_cutoffs]
plot_cumulative_probability(
    clean_core_data["CPORE"], clean_core_data["CPERM"], cutoffs=log_fzi_cutoffs
)
plot_modified_lorenz(clean_core_data["CPORE"], clean_core_data["CPERM"])


plot_fzi_histogram(clean_core_data["CPORE"], clean_core_data["CPERM"])

rt, stats = cluster_fzi(all_data.CPORE, all_data.CPERM, n_clusters=4)
plot_fzi(all_data["CPORE"], all_data["CPERM"], rock_type=rt, cut_offs=fzi_cutoffs)

print(fzi_cutoffs)
with open(rf"data\04_project\{project_name}\outputs\fzi_cutoffs.json", "w") as file:
    json.dump(fzi_cutoffs, file)

In [ ]:
from quick_pp.rock_type import plot_fzi, calc_fzi, rock_typing, plot_ward_dendogram

# Estimate rock types
fzi = calc_fzi(all_data["CPORE"], all_data["CPERM"])
all_data["FZI"] = fzi
fzi_rock_flag = rock_typing(fzi, fzi_cutoffs, higher_is_better=True)
all_data["ROCK_FLAG"] = fzi_rock_flag

plot_fzi(
    all_data["CPORE"], all_data["CPERM"], rock_type=fzi_rock_flag, cut_offs=fzi_cutoffs
)
plot_ward_dendogram(all_data["FZI"], p=30)
print(pd.Series(fzi_rock_flag).value_counts().sort_index())

### Develop Machine Learning models to predict ROCK_FLAG and FZI

The models will then be used to predict the Rock Type and FZI at non-cored intervals

In [ ]:
from quick_pp.rock_type import train_classification_model, train_regression_model
from quick_pp.machine_learning.feature_engineering import perform_rrt_smote

clean_core_data["PGF"] = clean_core_data.PHIE / clean_core_data.VCLAY
input_features = ["GR", "RHOB", "VCLAY", "PHIE", "PGF", "NPHI", "RT"]
train_data = perform_rrt_smote(clean_core_data, input_features + ["FZI"])
train_data = train_data.dropna(subset=input_features + ["ROCK_FLAG"])

fzi_rt_model = train_classification_model(
    train_data,
    input_features=input_features,
    target_feature="ROCK_FLAG",
    stratifier=train_data["ROCK_FLAG"],
)
with open(rf"data\04_project\{project_name}\outputs\fzi_rt_model.qppm", "wb") as file:
    pickle.dump(fzi_rt_model, file)

In [ ]:
train_data["ROCK_PRED"] = fzi_rt_model.predict(train_data[input_features])
input_features_fzi = input_features + ["ROCK_PRED"]
train_data["LOG_FZI"] = np.log10(train_data["FZI"])
fzi_model = train_regression_model(
    train_data,
    input_features=input_features_fzi,
    target_feature="LOG_FZI",
    stratifier=train_data["ROCK_PRED"],
)
with open(rf"data\04_project\{project_name}\outputs\fzi_model.qppm", "wb") as file:
    pickle.dump(fzi_model, file)

In [ ]:
from quick_pp.rock_type import plot_fzi, calc_fzi, rock_typing, plot_ward_dendogram

all_data["PGF"] = all_data.PHIE / all_data.VCLAY
all_data["GR_RATIO"] = all_data.GR.min() / all_data.GR.max()
predicted_fzi_rt = fzi_rt_model.predict(all_data[input_features])
plot_fzi(
    all_data["CPORE"],
    all_data["CPERM"],
    rock_type=predicted_fzi_rt,
    cut_offs=fzi_cutoffs,
)

In [ ]:
all_data["ROCK_PRED"] = fzi_rt_model.predict(all_data[input_features])

pred_data = all_data.dropna(subset=input_features_fzi)
predicted_fzi = 10 ** (fzi_model.predict(pred_data[input_features_fzi]))
predicted_fzi_rt = rock_typing(predicted_fzi, fzi_cutoffs, higher_is_better=True)
plot_fzi(
    pred_data["CPORE"],
    pred_data["CPERM"],
    rock_type=predicted_fzi_rt,
    cut_offs=fzi_cutoffs,
)

Comparing the predicted electrofacies with the clustered from lithofacies. The numbering of the clustered lithofacies is not following a particular order, thus it's mainly for comparing the cluster against the predicted values.

***
# Permeability Transform

### Determining the perm transform parameter for each rock type.

In [ ]:
from ipywidgets import interact, widgets

from quick_pp.core_calibration import poroperm_xplot

rt = widgets.Dropdown(
    options=sorted(all_data["ROCK_FLAG"].dropna().unique()),
    value=1,
    description="Rock Type:",
)
a = widgets.IntText(value=0, step=5000, description="a:")
b = widgets.FloatText(value=0, step=0.1, description="b:")


@interact(rt=rt, a=a, b=b)
def param(rt, a, b):
    data = all_data.copy()
    data["GROUP"] = np.where(data["ROCK_FLAG"] == rt, rt, 0)
    for group, group_data in data.groupby("GROUP"):
        label = f"RT{rt}:\n a={a:d},\n b={b:.2f}" if group == rt else ""
        poroperm_xplot(group_data["CPORE"], group_data["CPERM"], a=a, b=b, label=label)

In [ ]:
import pprint

from quick_pp.core_calibration import fit_poroperm_curve

poroperm_params = {}

for rt, data in all_data.groupby("ROCK_FLAG"):
    a, b = fit_poroperm_curve(data["CPORE"], data["CPERM"])
    poroperm_params[rt] = (a, b)

pp = pprint.PrettyPrinter(indent=4)
pp.pprint(poroperm_params)

In [ ]:
plt.figure(figsize=(12, 8))
for rt, data in all_data.groupby("ROCK_FLAG"):
    a, b = poroperm_params[rt]
    poroperm_xplot(
        data["CPORE"], data["CPERM"], a=a, b=b, label=f"RT{rt}: a={a:1d}, b={b:.2f}"
    )

In [ ]:
# Update poroperm_params with user input if needed
poroperm_params[4] = (4000, 4)

plt.figure(figsize=(12, 8))
for rt, data in all_data.groupby("ROCK_FLAG"):
    a, b = poroperm_params[rt]
    poroperm_xplot(
        data["CPORE"], data["CPERM"], a=a, b=b, label=f"RT{rt}: a={a:1d}, b={b:.2f}"
    )

### Compare different PERM estimations at well level

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, r2_score

from quick_pp.core_calibration import perm_transform
from quick_pp.rock_type import calc_fzi_perm

focus_well = "30-7a-7"
well_data = all_data[all_data.WELL_NAME == focus_well].copy()

# Permeability estimation based on core data perm transform
a = (
    well_data["ROCK_FLAG"]
    .map(poroperm_params)
    .apply(lambda x: x[0] if type(x) == tuple else np.nan)
)
b = (
    well_data["ROCK_FLAG"]
    .map(poroperm_params)
    .apply(lambda x: x[1] if type(x) == tuple else np.nan)
)
perm_trans = perm_transform(well_data["PHIE"], a=a, b=b)

# Permeability estimation based on FZI from core data
fzi = calc_fzi(well_data["CPORE"], well_data["CPERM"])
perm_fzi = calc_fzi_perm(fzi, well_data["PHIE"])

# Permeability prediction based on ROCK_FLAG ML model followed by perm transform

with open(rf"data\04_project\{project_name}\outputs\fzi_rt_model.qppm", "rb") as file:
    fzi_rt_model = pickle.load(file)
rock_flag_ml = fzi_rt_model.predict(well_data[input_features])
perm_a_ml = (
    pd.Series(rock_flag_ml)
    .map(poroperm_params)
    .apply(lambda x: x[0] if type(x) == tuple else np.nan)
)
perm_b_ml = (
    pd.Series(rock_flag_ml)
    .map(poroperm_params)
    .apply(lambda x: x[1] if type(x) == tuple else np.nan)
)
perm_ml = perm_transform(well_data["PHIE"], perm_a_ml, perm_b_ml)

# Permeability prediction based on FZI ML model followed by back calculate from FZI
well_data["ROCK_PRED"] = fzi_rt_model.predict(well_data[input_features])
with open(rf"data\04_project\{project_name}\outputs\fzi_model.qppm", "rb") as file:
    fzi_model = pickle.load(file)
fzi_ml = 10 ** (fzi_model.predict(well_data[input_features_fzi].ffill().bfill()))
perm_fzi_ml = calc_fzi_perm(fzi_ml, well_data["PHIE"])

# Plot to compare
plt.figure(figsize=(20, 2))
plt.plot(well_data.DEPTH, perm_trans, label="Perm Transform")
plt.plot(well_data.DEPTH, perm_fzi, label="Perm FZI")
# plt.plot(well_data.DEPTH, perm_ml, label='Perm ML')
plt.plot(well_data.DEPTH, perm_fzi_ml, label="Perm FZI ML")
plt.scatter(well_data.DEPTH, well_data.CPERM, label="Core Perm", marker=".", c="black")
plt.yscale("log")
plt.ylim(1e-3, 1e3)
plt.legend()

### Compare different ROCK_FLAG at well level

In [ ]:
# Compare rock types predicted from rt_model with applied cut-offs on predicted r35
rock_flag_fzi_ml = rock_typing(fzi_ml, fzi_cutoffs, higher_is_better=True)

plt.figure(figsize=(20, 2))
plt.plot(well_data.DEPTH, well_data.ROCK_FLAG, label="Rock Flag Core")
plt.plot(well_data.DEPTH, rock_flag_ml, label="Rock Flag ML")
plt.plot(well_data.DEPTH, rock_flag_fzi_ml, label="Rock Flag FZI ML")
plt.legend()

***
# Plotting the result

In [ ]:
from quick_pp.plotter.plotter import plotly_log

# Plot individual results
well_data["PERM"] = perm_fzi_ml
fig = plotly_log(well_data, well_name=focus_well, depth_uom="m")
fig.show(config=dict(scrollZoom=True))

# Apply to all

In [ ]:
from sklearn.impute import SimpleImputer
import json
from tqdm import tqdm

from quick_pp.rock_type import *

# Load FZI ROCK_FLAG
with open(rf"data\04_project\{project_name}\outputs\fzi_rt_model.qppm", "rb") as file:
    fzi_rt_model = pickle.load(file)

# Load FZI model
with open(rf"data\04_project\{project_name}\outputs\fzi_model.qppm", "rb") as file:
    fzi_model = pickle.load(file)

# Load FZI cutoffs
with open(rf"data\04_project\{project_name}\outputs\fzi_cutoffs.json", "rb") as file:
    fzi_cutoffs = json.load(file)

imp_mean = SimpleImputer(missing_values=np.nan, strategy="mean")

all_data["PGF"] = all_data.PHIE / all_data.VCLAY
input_features = ["GR", "RHOB", "VCLAY", "PHIE", "PGF", "NPHI", "RT"]
input_features_fzi = input_features + ["ROCK_PRED"]

for well_name, well_data in tqdm(
    all_data.groupby("WELL_NAME"), desc="Estimating for all wells"
):
    tqdm.write(f"Processing {well_name}: {len(well_data)} rows")

    rock_flag_ml = fzi_rt_model.predict(well_data[input_features])

    # # Predict permeability
    # temp_df = well_data.copy()
    # temp_df["ROCK_PRED"] = rock_flag_ml
    # temp_df[input_features] = imp_mean.fit_transform(temp_df[input_features])
    # fzi_ml = 10 ** (fzi_model.predict(temp_df[input_features_fzi]))
    # perm_ml = calc_fzi_perm(fzi_ml, well_data["PHIE"]).clip(lower=1e-3)

    # # Update rock type
    # rock_flag_ml = rock_typing(fzi_ml, cut_offs=fzi_cutoffs)

    # well_data["PERM"] = perm_ml
    well_data["ROCK_FLAG"] = rock_flag_ml

    # Save result to database
    with db_conn.get_session() as db_session:
        project = Project(db_session, name=project_name)
        project.update_data(well_data)
        project.save()

In [ ]:
with db_conn.get_session() as db_session:
    project = Project(db_session, name=project_name)
    df = project.get_all_data()
score_df = df[["WELL_NAME", "CPERM", "PERM"]].copy()
score_df.dropna(inplace=True)
mape = round(mean_absolute_percentage_error(score_df.CPERM, score_df.PERM), 2)
r2 = round(r2_score(score_df.CPERM, score_df.PERM), 2)
print(f"\n ### PERM MAPE: {mape:.2f}")
print(f" ### PERM R2: {r2:.2f}")

plt.scatter(score_df.CPERM, score_df.PERM, label=f"Overall - R2: {r2}, MAPE: {mape}")
for well, data in score_df.groupby("WELL_NAME"):
    mape = round(mean_absolute_percentage_error(data.CPERM, data.PERM), 2)
    r2 = round(r2_score(data.CPERM, data.PERM), 2)
    plt.scatter(data.CPERM, data.PERM, label=f"{well} - R2: {r2}, MAPE: {mape}")
plt.xlabel("Actual")
plt.ylabel("Calculated")
plt.xlim(1e-2, 1e5)
plt.ylim(1e-2, 1e5)
plt.loglog()
plt.legend()

In [ ]:
well_name = "30-7a-6"
copy_df = df[df.WELL_NAME == well_name]
plt.figure(figsize=(20, 3))
plt.scatter(copy_df.DEPTH, copy_df.CPERM, marker="x", c="red", label="CPERM")
plt.scatter(copy_df.DEPTH, copy_df.CPERM_ORI, marker=".", c="orange", label="CPERM ORI")
plt.plot(copy_df.DEPTH, copy_df.PERM, "b--", label="PERM")
plt.legend()
plt.ylim(1e-2, 1e4)
plt.yscale("log")
plt.xlim(3060, 3110)